# Model konteks untuk head MuQ saja

Buka dari root **SongMap**; memakai Python `.venv-songformer311` di sibling Section-Annotations.

Head MuQ saja (window 120 detik) lolos ablation: akurasi 0,7783 dan macro F1 0,7428, setara kontrol `all4` yang dilatih sama (0,7783 / 0,7438). Pipeline aplikasi memakai head **ditambah model konteks section**, dan model konteks lama memakai probabilitas `worship-v3` serta fitur audio 4096. Jadi model konteks harus dilatih ulang untuk kedua head dengan resep konteks yang sama persis:

1. `all4` (window 120) + konteks → pembanding berpasangan.
2. `muq` + konteks (fitur audio 2048: MuQ30 + MuQ120).

Aturan keputusan (ditetapkan sebelum hasil): `muq` + konteks menjadi *kandidat* bila akurasi dan macro F1 validation masing-masing turun maksimal 0,01 dibanding (a) `all4` + konteks, **dan** (b) pipeline produksi sekarang pada fitur asli: 0,7938 / 0,7755. Split test tidak dibaca di sini. Encoder tidak dijalankan; hanya cache fitur.

In [ ]:
from pathlib import Path
import subprocess, json
assert Path('scripts/train_variant_context.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=Path('../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
# Folder hasil percobaan kedua (window 120). Default: yang terbaru; ganti bila perlu.
ABLATION=sorted(Path('outputs.local').glob('stream-ablation-w120-*'))[-1].resolve()
HEADS={v:ABLATION/v/'best.pt' for v in ('all4','muq')}
for v,p in HEADS.items(): assert p.is_file(), f'Head {v} tidak ditemukan: {p}'
assert json.loads((ABLATION/'comparison.json').read_text())['variants']['muq']['comparison']['verdict']=='candidate'
OUTPUT=ABLATION/'context'
CACHE=Path('outputs.local')/'variant-context-cache'
DEVICE='mps'
print('Ablation:',ABLATION); print('Output:',OUTPUT.resolve())

In [ ]:
def run(*arguments,name):
    OUTPUT.mkdir(parents=True,exist_ok=True)
    log=OUTPUT/(name+'.log')
    command=[str(RUN_PYTHON),'-u','scripts/train_variant_context.py',*arguments]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log.resolve()}')

def train(variant,smoke=False):
    run('train','--head-checkpoint',str(HEADS[variant]),'--output-dir',str(OUTPUT/variant),
        '--cache-dir',str(CACHE),'--songstruct-root',str(SONGSTRUCT_ROOT),'--device',DEVICE,
        *(['--smoke-test'] if smoke else []),name=variant+('-smoke' if smoke else ''))

for variant in ('all4','muq'):
    train(variant,smoke=True)

Run penuh. Tiap varian: prediksi head untuk 32 lagu validation (harus mereproduksi angka validation head-nya) dan 251 lagu train, pooling audio per section (sekali, lalu di-cache), kemudian training konteks dengan early stopping. Prediksi head dan cache pooling dipakai ulang bila sel dijalankan lagi.

In [ ]:
train('all4')
train('muq')

In [ ]:
run('compare','--control',str(OUTPUT/'all4'),'--variant',str(OUTPUT/'muq'),name='compare')
print('Bagikan:',(OUTPUT/'muq'/'comparison.json').resolve())

Bila `muq` menjadi *kandidat*, langkah berikutnya: satu kali evaluasi split test untuk pipeline `muq` + konteks (dibandingkan dengan pipeline produksi pada test), lalu ekspor Core ML (head input 2048, palette6; konteks FP32) dan probe iPhone **tanpa MusicFM** untuk mengukur ukuran dan waktu sebenarnya. Bila *ditolak*, kedua encoder tetap dipakai dan jalur berikutnya adalah kompresi encoder di bawah 6 bit.